In [0]:
# ============================================================
# SENTINEL COMMERCE
# Notebook: 01_generate_source_data
# Purpose : Generate realistic raw order files for ingestion
# ============================================================

from datetime import datetime, timedelta, timezone
import random
import uuid
import json

landing_path = "/Volumes/sentinel_dev/landing/source_files/orders"


print(f"Landing path: {landing_path}")
print("Adding commit")

In [0]:
dbutils.fs.mkdirs(landing_path)

print("Landing directory created.")

In [0]:
NUM_FILES = 20
RECORDS_PER_FILE = 50

customer_ids = [f"CUST-{i:05d}" for i in range(1, 501)]

products = [
    ("PROD-001", "Wireless Headphones", 2999.00),
    ("PROD-002", "Mechanical Keyboard", 4499.00),
    ("PROD-003", "USB-C Hub", 1899.00),
    ("PROD-004", "Laptop Stand", 2499.00),
    ("PROD-005", "Webcam", 3499.00),
    ("PROD-006", "Gaming Mouse", 2199.00),
    ("PROD-007", "Monitor Light Bar", 3999.00),
    ("PROD-008", "Desk Mat", 999.00),
]

payment_methods = [
    "UPI",
    "CREDIT_CARD",
    "DEBIT_CARD",
    "NET_BANKING",
    "COD"
]

order_statuses = [
    "PLACED",
    "CONFIRMED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED"
]

In [0]:
def generate_order():
    product_id, product_name, unit_price = random.choice(products)

    quantity = random.randint(1, 4)

    return {
        "order_id": str(uuid.uuid4()),
        "customer_id": random.choice(customer_ids),

        "product_id": product_id,
        "product_name": product_name,

        "quantity": quantity,
        "unit_price": unit_price,
        "total_amount": round(quantity * unit_price, 2),

        "payment_method": random.choice(payment_methods),
        "order_status": random.choice(order_statuses),

        "order_timestamp": (
            datetime.now(timezone.utc)
            - timedelta(minutes=random.randint(0, 1440))
        ).isoformat(),

        "source_system": "sentinel_web"
    }

In [0]:
for file_number in range(NUM_FILES):

    records = [
        generate_order()
        for _ in range(RECORDS_PER_FILE)
    ]

    file_name = (
        f"{landing_path}/"
        f"orders_{datetime.now().strftime('%Y%m%d_%H%M%S')}_"
        f"{file_number:03d}.json"
    )

    file_content = "\n".join(
        json.dumps(record)
        for record in records
    )

    dbutils.fs.put(
        file_name,
        file_content,
        overwrite=True
    )

print(
    f"Generated {NUM_FILES} files "
    f"with {RECORDS_PER_FILE} records each."
)

In [0]:
files = dbutils.fs.ls(landing_path)

display(files)

In [0]:
sample_file = files[0].path

print(dbutils.fs.head(sample_file, 3000))

In [0]:
# ============================================================
# SCHEMA EVOLUTION TEST
# Simulate a new source application version
# ============================================================

new_records = []

for _ in range(100):

    order = generate_order()

    # New fields introduced by upstream system
    order["coupon_code"] = random.choice([
        "WELCOME20",
        "SUMMER10",
        "VIP25",
        None
    ])

    order["device_type"] = random.choice([
        "MOBILE",
        "DESKTOP",
        "TABLET"
    ])

    new_records.append(order)


file_name = (
    f"{landing_path}/"
    f"orders_schema_v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

file_content = "\n".join(
    json.dumps(record)
    for record in new_records
)

dbutils.fs.put(
    file_name,
    file_content,
    overwrite=True
)

print("Generated schema-v2 file with 100 records.")
print(f"File: {file_name}")

In [0]:
# ============================================================
# RESCUED DATA TEST
# Simulate malformed / structurally inconsistent source data
# ============================================================

bad_records = [
    {
        "order_id": "BAD-ORDER-001",
        "customer_id": "CUST-00001",
        "product_id": "PROD-001",
        "product_name": "Wireless Headphones",
        "quantity": 1,

        # Expected scalar value, but upstream sends an object
        "unit_price": {
            "value": 2999,
            "currency": "INR"
        },

        "total_amount": 2999,
        "payment_method": "UPI",
        "order_status": "PLACED",
        "order_timestamp": datetime.now(timezone.utc).isoformat(),
        "source_system": "broken_partner_api"
    },

    {
        "order_id": "BAD-ORDER-002",
        "customer_id": "CUST-00002",
        "product_id": "PROD-003",
        "product_name": "USB-C Hub",

        # Expected number-like value, but upstream sends nested data
        "quantity": {
            "ordered": 2,
            "backordered": 1
        },

        "unit_price": 1899,
        "total_amount": 3798,
        "payment_method": "CREDIT_CARD",
        "order_status": "CONFIRMED",
        "order_timestamp": datetime.now(timezone.utc).isoformat(),
        "source_system": "broken_partner_api"
    }
]

bad_file_name = (
    f"{landing_path}/"
    f"orders_bad_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

bad_file_content = "\n".join(
    json.dumps(record)
    for record in bad_records
)

dbutils.fs.put(
    bad_file_name,
    bad_file_content,
    overwrite=True
)

print(f"Generated bad-data file: {bad_file_name}")

In [0]:
# ============================================================
# DUPLICATE / UPDATE TEST
#
# Simulates multiple versions of the same business order.
# ============================================================

duplicate_order_id = "DUP-ORDER-001"

duplicate_records = [
    {
        "order_id": duplicate_order_id,
        "customer_id": "CUST-00250",
        "product_id": "PROD-003",
        "product_name": "USB-C Hub",
        "quantity": 1,
        "unit_price": 1899,
        "total_amount": 1899,
        "payment_method": "UPI",
        "order_status": "PLACED",
        "order_timestamp": (
            datetime.now(timezone.utc)
            - timedelta(hours=3)
        ).isoformat(),
        "source_system": "sentinel_web"
    },

    {
        "order_id": duplicate_order_id,
        "customer_id": "CUST-00250",
        "product_id": "PROD-003",
        "product_name": "USB-C Hub",
        "quantity": 1,
        "unit_price": 1899,
        "total_amount": 1899,
        "payment_method": "UPI",
        "order_status": "CONFIRMED",
        "order_timestamp": (
            datetime.now(timezone.utc)
            - timedelta(hours=2)
        ).isoformat(),
        "source_system": "sentinel_web"
    },

    {
        "order_id": duplicate_order_id,
        "customer_id": "CUST-00250",
        "product_id": "PROD-003",
        "product_name": "USB-C Hub",
        "quantity": 1,
        "unit_price": 1899,
        "total_amount": 1899,
        "payment_method": "UPI",
        "order_status": "SHIPPED",
        "order_timestamp": (
            datetime.now(timezone.utc)
            - timedelta(hours=1)
        ).isoformat(),
        "source_system": "sentinel_web"
    }
]

In [0]:
duplicate_file = (
    f"{landing_path}/"
    f"orders_duplicate_test_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

duplicate_content = "\n".join(
    json.dumps(record)
    for record in duplicate_records
)

dbutils.fs.put(
    duplicate_file,
    duplicate_content,
    overwrite=True
)

print("Generated 3 versions of DUP-ORDER-001")

In [0]:
# ============================================================
# SILVER MERGE TEST
#
# Simulate a later update to an existing order.
# ============================================================

merge_test_record = {
    "order_id": "DUP-ORDER-001",
    "customer_id": "CUST-00250",
    "product_id": "PROD-003",
    "product_name": "USB-C Hub",

    "quantity": 1,
    "unit_price": 1899,
    "total_amount": 1899,

    "payment_method": "UPI",

    # Previously SHIPPED
    "order_status": "DELIVERED",

    "order_timestamp": datetime.now(timezone.utc).isoformat(),

    "source_system": "sentinel_web"
}

In [0]:
merge_test_file = (
    f"{landing_path}/"
    f"orders_merge_test_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

dbutils.fs.put(
    merge_test_file,
    json.dumps(merge_test_record),
    overwrite=True
)

print("Generated DELIVERED update for DUP-ORDER-001")

In [0]:
# ============================================================
# LATE-ARRIVING EVENT TEST
#
# Scenario:
# An event arrives AFTER the order is already DELIVERED,
# but its business timestamp belongs earlier in the lifecycle.
#
# Purpose:
# Validate event-time sequencing in Lakeflow AUTO CDC.
# ============================================================

from datetime import datetime, timezone
import json


late_event = {
    "order_id": "DUP-ORDER-001",
    "customer_id": "CUST-00250",
    "product_id": "PROD-003",
    "product_name": "USB-C Hub",

    "quantity": 1,
    "unit_price": 1899,
    "total_amount": 1899,

    "payment_method": "UPI",

    # Use an existing valid status so we're testing CDC,
    # not changing the DQ contract.
    "order_status": "CONFIRMED",

    # Deliberately BETWEEN the original CONFIRMED and SHIPPED.
    "order_timestamp": "2026-08-23T03:30:00+00:00",

    "source_system": "sentinel_web"
}

In [0]:
late_event_file = (
    f"{landing_path}/"
    f"orders_late_event_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

dbutils.fs.put(
    late_event_file,
    json.dumps(late_event),
    overwrite=True
)

print(f"Late-arriving event generated:")
print(late_event_file)

In [0]:
# ============================================================
# INCREMENTAL GOLD TEST
#
# 1 new order
# 1 update to an existing order
# ============================================================

from datetime import datetime, timezone, timedelta
import json

now = datetime.now(timezone.utc)

incremental_test_records = [

    # NEW ORDER → should INSERT into Gold
    {
        "order_id": "INC-ORDER-001",
        "customer_id": "CUST-00250",
        "product_id": "PROD-003",
        "product_name": "USB-C Hub",
        "quantity": 2,
        "unit_price": 1899,
        "total_amount": 3798,
        "payment_method": "UPI",
        "order_status": "PLACED",
        "order_timestamp": now.isoformat(),
        "source_system": "sentinel_web"
    },

    # EXISTING ORDER UPDATE → should UPDATE Gold
    {
        "order_id": "DUP-ORDER-001",
        "customer_id": "CUST-00250",
        "product_id": "PROD-003",
        "product_name": "USB-C Hub",
        "quantity": 1,
        "unit_price": 1899,
        "total_amount": 1899,
        "payment_method": "UPI",
        "order_status": "DELIVERED",
        "order_timestamp": (
            now + timedelta(minutes=1)
        ).isoformat(),
        "source_system": "sentinel_web"
    }
]

In [0]:
from datetime import datetime, timezone
import json

now = datetime.now(timezone.utc)

record = {
    "order_id": "INC-ORDER-002",
    "customer_id": "CUST-00250",
    "product_id": "PROD-003",
    "product_name": "USB-C Hub",
    "quantity": 2,
    "unit_price": 1899,
    "total_amount": 3798,
    "payment_method": "UPI",
    "order_status": "PLACED",
    "order_timestamp": now.isoformat(),
    "source_system": "sentinel_web"
}

file_path = (
    f"{landing_path}/"
    f"orders_incremental_002_"
    f"{now.strftime('%Y%m%d_%H%M%S')}.json"
)

dbutils.fs.put(
    file_path,
    json.dumps(record),
    overwrite=True
)

print(file_path)

In [0]:
from datetime import datetime, timezone
import json

now = datetime.now(timezone.utc)

record = {
    "order_id": "CDF-ORDER-001",
    "customer_id": "CUST-00250",
    "product_id": "PROD-003",
    "product_name": "USB-C Hub",
    "quantity": 3,
    "unit_price": 1899,
    "total_amount": 5697,
    "payment_method": "UPI",
    "order_status": "PLACED",
    "order_timestamp": now.isoformat(),
    "source_system": "sentinel_web"
}

file_path = (
    f"{landing_path}/"
    f"orders_cdf_test_"
    f"{now.strftime('%Y%m%d_%H%M%S')}.json"
)

dbutils.fs.put(
    file_path,
    json.dumps(record),
    overwrite=True
)

print(file_path)